# EDA — `incidents_pannes`

---

## Objectif de ce notebook

Ce notebook réalise l'analyse exploratoire de la table `incidents_pannes`.  
Il couvre l'inspection initiale, la répartition des types et gravités d'incidents,  
l'analyse financière et opérationnelle, les corrélations entre variables clés,  
et formule les décisions qui guideront le modèle de classification des incidents.

---

## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

os.makedirs("figures", exist_ok=True)

print("Imports OK ✅")

## 1. Chargement des Données

In [ ]:
df = pd.read_csv("../data/incidents_pannes.csv")

df['date_incident'] = pd.to_datetime(df['date_incident'])
df['date_resolution'] = pd.to_datetime(df['date_resolution'])
df['annee'] = df['date_incident'].dt.year
df['mois'] = df['date_incident'].dt.month

print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période    : {df['date_incident'].min().date()} → {df['date_incident'].max().date()}")
print(f"Dépôts     : {df['depot_id'].nunique()}")
print(f"Types d'incidents : {df['type_incident'].nunique()}")
print()
df.head(5)

## 2. Inspection Initiale

On examine les types de colonnes, les valeurs manquantes et les statistiques descriptives.

In [ ]:
print("=== Types de colonnes ===")
print(df.dtypes)
print()
print("=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante ✅")
print()
print(f"=== Doublons : {df.duplicated().sum()} ===")

In [ ]:
# Statistiques descriptives des variables numériques
df[['cout_incident_usd', 'duree_arret_heures', 'quantite_perdue']].describe().round(2)

In [ ]:
# Répartition des 4 niveaux de gravité et des 8 types d'incidents
print("=== Répartition par gravité ===")
gravite_counts = df['gravite'].value_counts()
for g, n in gravite_counts.items():
    print(f"  {g:<12} : {n:>4} ({n/len(df)*100:.1f}%)")

print()
print("=== Répartition par type d'incident ===")
type_counts = df['type_incident'].value_counts()
for t, n in type_counts.items():
    print(f"  {t:<30} : {n:>3} ({n/len(df)*100:.1f}%)")

**📝 Observations :**

> Le dataset contient **602 incidents** sur 10 ans (2015–2024), soit en moyenne **60 incidents par an** ou environ **5 par mois**. Aucune valeur manquante, aucun doublon — les données sont propres.
>
> La répartition par gravité révèle un **déséquilibre des classes** important, typique d'un jeu de données réel : 52,0% des incidents sont Faibles, 27,6% Modérés, 17,1% Élevés et seulement **3,3% Critiques** (20 incidents sur 602). Ce déséquilibre devra être traité explicitement lors de la modélisation (stratification, class_weight, SMOTE...).
>
> Les 8 types d'incidents sont répartis de façon relativement équilibrée (entre 66 et 82 occurrences chacun), ce qui est favorable pour la classification.

## 3. Répartition et Fréquence des Incidents

On visualise la distribution des types d'incidents, leur gravité, leur répartition par dépôt et leur évolution dans le temps.

In [ ]:
# Barplot horizontal des types d'incidents par fréquence
type_freq = df['type_incident'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
type_freq.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Fréquence des types d'incidents")
ax.set_xlabel("Nombre d'incidents")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figures/01_types_incidents.png", dpi=150)
plt.show()

In [ ]:
# Camembert de la répartition par gravité
gravite_order = ['Faible', 'Modéré', 'Élevé', 'Critique']
gravite_counts = df['gravite'].value_counts().reindex(gravite_order)
colors = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    gravite_counts,
    labels=gravite_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(12)
ax.set_title("Répartition des incidents par gravité", fontsize=13)
plt.tight_layout()
plt.savefig("figures/02_repartition_gravite.png", dpi=150)
plt.show()

In [ ]:
# Nombre d'incidents par dépôt
incidents_depot = df['depot_nom'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
incidents_depot.plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
ax.set_title("Nombre d'incidents par dépôt")
ax.set_xlabel("Nombre d'incidents")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figures/03_incidents_par_depot.png", dpi=150)
plt.show()

In [ ]:
# Évolution annuelle du nombre d'incidents sur 10 ans
incidents_annee = df.groupby('annee').size()

fig, ax = plt.subplots(figsize=(10, 5))
incidents_annee.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Évolution annuelle du nombre d'incidents (2015–2024)")
ax.set_xlabel("Année")
ax.set_ylabel("Nombre d'incidents")
ax.set_xticklabels(incidents_annee.index, rotation=0)
ax.axhline(incidents_annee.mean(), color='red', linestyle='--',
           linewidth=1.2, label=f"Moyenne : {incidents_annee.mean():.0f}")
ax.legend()
plt.tight_layout()
plt.savefig("figures/04_incidents_par_annee.png", dpi=150)
plt.show()

**📝 Observations :**

> Les 8 types d'incidents sont relativement équilibrés en fréquence. La **Contamination produit** (82 cas) et la **Panne électrique** (80 cas) sont les plus fréquents, tandis que la **Défaillance capteur** (66 cas) est la moins fréquente — mais c'est aussi l'une des plus coûteuses comme on le verra plus loin.
>
> Côté dépôts, le **Dépôt Sokodé** (85 incidents) et le **Dépôt Tsévié** (82) sont les plus touchés, tandis que le **Dépôt Central Lomé** (63) enregistre le moins d'incidents — ce qui est surprenant pour le dépôt le plus actif en volume, et mérite d'être noté.
>
> L'évolution annuelle ne montre pas de tendance claire à la hausse ou à la baisse — les incidents sont distribués de façon relativement stable autour de la moyenne annuelle de 60 incidents.

## 4. Analyse Financière et Opérationnelle

On analyse les coûts, les durées d'arrêt et les produits les plus impliqués.

In [ ]:
# Coût moyen par type d'incident
cout_type = df.groupby('type_incident')['cout_incident_usd'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
cout_type.plot(kind='bar', ax=ax, color='crimson', edgecolor='white')
ax.set_title("Coût moyen par type d'incident (USD)")
ax.set_xlabel("")
ax.set_ylabel("Coût moyen (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig("figures/05_cout_par_type.png", dpi=150)
plt.show()

In [ ]:
# Coût total des incidents par dépôt
cout_depot = df.groupby('depot_nom')['cout_incident_usd'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
cout_depot.plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
ax.set_title("Coût total des incidents par dépôt (USD)")
ax.set_xlabel("")
ax.set_ylabel("Coût total (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig("figures/06_cout_par_depot.png", dpi=150)
plt.show()

In [ ]:
# Durée d'arrêt moyenne par niveau de gravité
gravite_order = ['Faible', 'Modéré', 'Élevé', 'Critique']
duree_gravite = df.groupby('gravite')['duree_arret_heures'].mean().reindex(gravite_order)
colors = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(duree_gravite.index, duree_gravite.values,
              color=colors, edgecolor='white')
for bar, val in zip(bars, duree_gravite.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}h", ha='center', fontsize=11)
ax.set_title("Durée d'arrêt moyenne par niveau de gravité")
ax.set_xlabel("Gravité")
ax.set_ylabel("Durée d'arrêt (heures)")
plt.tight_layout()
plt.savefig("figures/07_duree_par_gravite.png", dpi=150)
plt.show()

print("Coût moyen par gravité :")
cout_gravite = df.groupby('gravite')['cout_incident_usd'].mean().reindex(gravite_order)
for g, c in cout_gravite.items():
    print(f"  {g:<10} : {c:>12,.0f} USD")

In [ ]:
# Produits les plus impliqués dans des incidents
produit_freq = df['produit_concerne_nom'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
produit_freq.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Produits les plus impliqués dans des incidents")
ax.set_xlabel("Nombre d'incidents")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figures/08_incidents_par_produit.png", dpi=150)
plt.show()

In [ ]:
# Taux de résolution global et par type d'incident
df['resolu'] = df['statut'].isin(['Résolu', 'Fermé'])
taux_global = df['resolu'].mean() * 100
print(f"Taux de résolution global : {taux_global:.1f}%")
print()

taux_par_type = df.groupby('type_incident')['resolu'].mean().sort_values() * 100

fig, ax = plt.subplots(figsize=(10, 5))
taux_par_type.plot(kind='barh', ax=ax,
                   color=['#4CAF50' if v >= 80 else '#FFC107' for v in taux_par_type],
                   edgecolor='white')
ax.axvline(80, color='red', linestyle='--', linewidth=1.2, label='Seuil 80%')
ax.set_title("Taux de résolution par type d'incident (%)")
ax.set_xlabel("Taux de résolution (%)")
ax.set_ylabel("")
ax.legend()
plt.tight_layout()
plt.savefig("figures/09_taux_resolution.png", dpi=150)
plt.show()

**📝 Observations :**

> L'analyse financière révèle des écarts considérables entre types d'incidents. La **Fuite de produit** est de loin la plus coûteuse en moyenne (1,95M USD), suivie de la **Défaillance capteur** (1,44M USD) — paradoxalement, ce type est peu fréquent mais très impactant financièrement. À l'inverse, la **Panne équipement** et la **Panne électrique** sont les moins coûteuses (~490–500K USD).
>
> La corrélation entre gravité et durée d'arrêt est très nette et attendue : un incident **Critique** immobilise le dépôt en moyenne **148,9 heures** (soit plus de 6 jours), contre seulement **2,5 heures** pour un incident Faible. Les coûts suivent la même logique : 5,24M USD en moyenne pour un incident Critique vs 293K USD pour un incident Faible — soit un rapport de **1 à 18**.
>
> Le taux de résolution global est de **84,2%**, ce qui est satisfaisant. Les incidents en cours ou en investigation (15,8%) sont principalement des incidents récents ou complexes.
>
> **Décision de modélisation :** la gravité est clairement la variable cible la plus pertinente pour la classification. Le déséquilibre des classes (52% Faible vs 3,3% Critique) impose d'utiliser `class_weight='balanced'` dans Random Forest et SVM, et de privilégier le **F1-score macro** plutôt que l'accuracy comme métrique principale.

## 5. Corrélations entre Variables Clés

On analyse les relations entre le coût, la quantité perdue et la durée d'arrêt.

In [ ]:
# Matrice de corrélation entre coût, quantité perdue et durée d'arrêt
cols_corr = ['cout_incident_usd', 'quantite_perdue', 'duree_arret_heures']
labels_corr = ['Coût (USD)', 'Quantité perdue', "Durée d'arrêt (h)"]

corr = df[cols_corr].corr()
corr.columns = labels_corr
corr.index = labels_corr

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax,
            annot_kws={'size': 13})
ax.set_title("Matrice de corrélation — Variables financières et opérationnelles")
plt.tight_layout()
plt.savefig("figures/10_correlation_incidents.png", dpi=150)
plt.show()

In [ ]:
# Scatter plot : durée d'arrêt vs coût, coloré par gravité
colors_map = {'Faible': '#4CAF50', 'Modéré': '#FFC107',
              'Élevé': '#FF5722', 'Critique': '#B71C1C'}

fig, ax = plt.subplots(figsize=(10, 6))
for gravite, group in df.groupby('gravite'):
    ax.scatter(
        group['duree_arret_heures'],
        group['cout_incident_usd'],
        label=gravite,
        color=colors_map[gravite],
        alpha=0.6, s=50, edgecolors='white', linewidth=0.5
    )
ax.set_title("Durée d'arrêt vs Coût de l'incident, par gravité")
ax.set_xlabel("Durée d'arrêt (heures)")
ax.set_ylabel("Coût (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax.legend(title='Gravité')
plt.tight_layout()
plt.savefig("figures/11_scatter_duree_cout.png", dpi=150)
plt.show()

**📝 Observations :**

> La matrice de corrélation révèle que la **durée d'arrêt** est la variable la plus corrélée au coût de l'incident — ce qui est logique : plus un arrêt dure, plus il coûte cher en pertes de production et en interventions. La **quantité perdue** est moins corrélée au coût que la durée, ce qui s'explique par la diversité des unités (litres, tonnes, barils) et des valeurs unitaires selon les produits.
>
> Le scatter plot confirme visuellement la séparation des classes par gravité : les incidents **Critiques** (rouge foncé) se concentrent dans le quadrant supérieur droit (durée longue, coût élevé), tandis que les incidents **Faibles** (vert) restent groupés en bas à gauche. Cette séparation visuelle est un signal positif pour la classification supervisée.
>
> **Décision de modélisation :** `duree_arret_heures` et `cout_incident_usd` seront des variables d'entrée importantes pour le modèle. Attention cependant : dans un vrai système, le coût n'est connu qu'après résolution de l'incident — il ne peut pas être utilisé pour classifier un incident **au moment de sa déclaration**. Il sera donc **exclu des variables d'entrée** du modèle de classification pour éviter une fuite de données (data leakage).

## 6. Conclusions et Décisions

---

### 🔍 Observations clés

> 1. **602 incidents** sur 10 ans, répartis équitablement entre les 8 dépôts et les 8 types — pas de concentration géographique ou par type.
> 2. **Déséquilibre des classes sévère** : 52,0% Faible vs 3,3% Critique — stratégie de rééquilibrage obligatoire.
> 3. La **Fuite de produit** est le type le plus coûteux (1,95M USD en moyenne) malgré une fréquence moyenne — c'est le type à surveiller en priorité.
> 4. La **durée d'arrêt** est le meilleur indicateur du coût : un incident Critique dure en moyenne 148,9h contre 2,5h pour un incident Faible.
> 5. Le taux de résolution est de **84,2%** — satisfaisant mais 15,8% des incidents restent ouverts.
> 6. Le **coût** ne peut pas être utilisé comme variable d'entrée du modèle (disponible uniquement après résolution).

---

### ✅ Décisions pour la modélisation (Notebook 08)

> - **Variable cible** : `gravite` (4 classes : Faible / Modéré / Élevé / Critique)
> - **Variables d'entrée retenues** : `type_incident`, `depot_id`, `produit_concerne_id`, `duree_arret_heures`, `quantite_perdue`, `heure_incident` (heure de la journée), `mois`
> - **Variables exclues** : `cout_incident_usd` (data leakage), `description`, `mesures_correctives` (texte libre difficile à encoder simplement)
> - **Gestion du déséquilibre** : `class_weight='balanced'` pour Random Forest et SVM
> - **Métrique principale** : F1-score macro (pénalise les mauvaises prédictions sur les classes minoritaires)
> - **Seuil d'escalade proposé** : tout incident classifié **Élevé ou Critique** déclenche une notification immédiate au responsable de dépôt

---

### 📁 Figures produites

| Fichier | Description |
|---|---|
| `01_types_incidents.png` | Fréquence des types d'incidents |
| `02_repartition_gravite.png` | Camembert répartition par gravité |
| `03_incidents_par_depot.png` | Incidents par dépôt |
| `04_incidents_par_annee.png` | Évolution annuelle |
| `05_cout_par_type.png` | Coût moyen par type |
| `06_cout_par_depot.png` | Coût total par dépôt |
| `07_duree_par_gravite.png` | Durée d'arrêt par gravité |
| `08_incidents_par_produit.png` | Produits impliqués |
| `09_taux_resolution.png` | Taux de résolution par type |
| `10_correlation_incidents.png` | Matrice de corrélation |
| `11_scatter_duree_cout.png` | Scatter durée vs coût par gravité |